# Statistical Analysis

## Objective

While Exploratory Data Analysis identifies patterns visually, statistical analysis validates whether those patterns are statistically significant.

In this notebook, we will answer the following business questions:

1. What are the statistical characteristics of key business variables?
2. Does late delivery significantly affect customer ratings?
3. Does average payment value differ across payment methods?
4. Is there a significant relationship between product price and customer ratings?
5. What is the 95% confidence interval for the average review score?

In [1]:
import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns

from scipy import stats
from scipy.stats import ttest_ind
from scipy.stats import f_oneway
from scipy.stats import pearsonr

plt.style.use("ggplot")

In [2]:
master_df = pd.read_csv("../Data/master_dataset.csv")

In [3]:
date_columns = [
    "order_purchase_timestamp",
    "order_approved_at",
    "order_delivered_customer_date",
    "order_estimated_delivery_date"
]

for col in date_columns:
    if col in master_df.columns:
        master_df[col] = pd.to_datetime(master_df[col], errors="coerce")

In [4]:
master_df.shape

(119143, 49)

## 1. Descriptive Statistics

Let's summarize the major business variables.

In [5]:
stats_df = master_df[
    [
        "price",
        "payment_value",
        "delivery_time",
        "delivery_delay",
        "review_score"
    ]
]

stats_df.describe().T

,count,mean,std,min,25%,50%,75%,max
price,118310.0,120.646603,184.109691,0.85,39.90,74.90,134.90,6735.00
payment_value,119140.0,172.735135,267.776077,0.00,60.85,108.16,189.24,13664.08
delivery_time,115722.0,12.022589,9.454922,0.00,6.00,10.00,15.00,209.00
delivery_delay,115722.0,-12.048392,10.163801,-147.00,-17.00,-13.00,-7.00,188.00
review_score,118146.0,4.015582,1.400436,1.00,4.00,5.00,5.00,5.00


## Business Question

Does late delivery significantly affect customer ratings?

### H₀

Late delivery does not affect review scores.

### H₁

Late delivery affects review scores.

In [8]:
on_time = master_df.loc[
    master_df["late_delivery"] == False,
    "review_score"
].dropna()

late = master_df.loc[
    master_df["late_delivery"] == True,
    "review_score"
].dropna()

t_stat, p_value = ttest_ind(on_time, late)

print(f"T-statistic : {t_stat:.3f}")
print(f"P-value     : {p_value:.6f}")

T-statistic : 117.926
P-value     : 0.000000


In [9]:
if p_value < 0.05:
    print("Reject H₀")
    print("Late delivery significantly affects customer ratings.")
else:
    print("Fail to reject H₀")

Reject H₀
Late delivery significantly affects customer ratings.


## Business Question

Does payment value differ across payment methods?

### H₀

Average payment value is the same across all payment methods.

### H₁

At least one payment method has a different average payment value.

In [10]:
groups = [
    group["payment_value"].values
    for _, group in master_df.groupby("payment_type")
]

f_stat, p_value = f_oneway(*groups)

print(f"F-statistic : {f_stat:.2f}")
print(f"P-value     : {p_value:.6f}")

F-statistic : 272.19
P-value     : 0.000000


In [11]:
if p_value < 0.05:
    print("Reject H₀")
    print("Payment values differ significantly across payment methods.")
else:
    print("Fail to reject H₀")

Reject H₀
Payment values differ significantly across payment methods.


## Business Question

Is product price correlated with customer review score?

In [12]:
corr_df = master_df[
    ["price","review_score"]
].dropna()

r,p = pearsonr(
    corr_df["price"],
    corr_df["review_score"]
)

print(f"Correlation : {r:.3f}")
print(f"P-value     : {p:.6f}")

Correlation : -0.004
P-value     : 0.123903


In [13]:
if p < 0.05:
    print("Statistically significant correlation.")
else:
    print("No statistically significant correlation.")

No statistically significant correlation.


## 95% Confidence Interval for Average Review Score

In [14]:
review = master_df["review_score"].dropna()

mean = review.mean()

std = review.std()

n = len(review)

margin = stats.t.ppf(
    0.975,
    df=n-1
) * std / np.sqrt(n)

lower = mean - margin

upper = mean + margin

print(f"Mean Review Score : {mean:.3f}")

print(f"95% Confidence Interval : ({lower:.3f}, {upper:.3f})")

Mean Review Score : 4.016
95% Confidence Interval : (4.008, 4.024)


# Conclusion

## Key Statistical Findings

- Payment values exhibit substantial variability and contain high-value outliers.
- Late deliveries significantly reduce customer ratings (t-test).
- Payment values differ significantly across payment methods (ANOVA).
- Product price has little or no meaningful relationship with customer ratings (Pearson correlation).
- The average customer review score can be estimated with a narrow 95% confidence interval, indicating a stable overall satisfaction level.

Overall, the statistical analysis supports the business insights obtained during the exploratory data analysis and provides quantitative evidence for decision-making.